# Download NISAR SME2 soil moisture products (Alaska)

Downloads NISAR L3 SME2 (`NISAR_L3_PR_SME2_*.h5`, ~200m EASE-Grid 2.0 soil
moisture) for the Alaska ISMN stations, over each station's own NISAR
acquisition date range -- the Alaska counterpart to the `SME2_data/`
directory `NISAR_SM.ipynb` (Spain) already reads and
[`SME2_footprint_check.ipynb`](SME2_footprint_check.ipynb) inspects. Neither
of those notebooks downloads SME2 themselves -- this is the actual download
step, missing until now.

**Search**: `find_nisar` below (inlined from `nisar_pytools`'s
`io/search.py` + `utils/search_validation.py` -- that was the only function
from that ~5,000-line vendored package this project ever used, so the
package itself was dropped rather than kept around as a dependency for two
functions), which queries ASF/CMR metadata directly -- no auth needed for
search, and confirmed working against the live catalog. `PRODUCT_TYPES`
includes `"SME2"` (`asf_search` itself already defines `PRODUCT_TYPE.SME2`).

**Download**: `earthaccess.download()`, *not* a `download_urls()`-style
plain-`requests` fetch. These URLs are on NASA's Earthdata **Cloud** (the
`nisar.asf.earthdatacloud.nasa.gov` host), which needs a real Earthdata Login
OAuth/bearer-token handshake to fetch bytes from outside AWS us-west-2 --
plain unauthenticated `requests` (relying on a `.netrc` entry that turned out
not to exist on this machine) fails on every single file with no way to
complete that handshake, confirmed by testing both directly. `earthaccess` is
the tool already used successfully against this exact host elsewhere in this
project (`Earsel26_NISAR-main`'s notebooks 02/03), so Section 3 uses it here
too -- `provider="ASF"`, confirmed against this data's own CMR metadata
(`meta.provider-id == "ASF"`).

**Search scope**: per station, not a single shared window or track/frame
filter -- Alaska stations are covered by several different NISAR relative
orbits (same reason `S1_burst_stations.ipynb`/`NISAR_SM_alaska.ipynb` never
filter to one track), so this reuses that notebook's own
`load_station_acquisitions`-based per-station date range (padded) and 80m
station-footprint bounding box, letting CMR return SME2 granules from
whichever track(s) actually cover each station.

**Prerequisite**: a NASA Earthdata Login account. `earthaccess.login()`
(Section 3) will prompt for username/password interactively the first time
this runs in a given kernel session if no cached credentials/`.netrc` are
found -- this is expected, not an error; enter your Earthdata credentials
when asked. Each SME2 granule is also fairly large (~125MB observed from
real CMR metadata), so expect real bandwidth/storage use across ~15-20
stations x several tracks each, not a trivial download.

In [ ]:
import sys
import os
from pathlib import Path

import pyproj
pyproj.datadir.set_data_dir(os.path.join(sys.prefix, "Library", "share", "proj"))


def require_globals(*names):
    missing = [n for n in names if n not in globals()]
    if missing:
        raise RuntimeError(
            f"Missing {missing} -- run the earlier setup cells first "
            "(the kernel may have been restarted)."
        )


BASE = Path.cwd()
EARSEL_NOTEBOOKS = BASE / "Earsel26_NISAR-main" / "notebooks"
EARSEL_ROOT = EARSEL_NOTEBOOKS.parent
GSLC_WINDOWS_DIR = EARSEL_NOTEBOOKS / "nisar_data" / "alaska_GSLC_windows"
SME2_OUT_DIR = BASE / "SME2_data_alaska"

if str(EARSEL_ROOT) not in sys.path:
    sys.path.append(str(EARSEL_ROOT))

import json
import logging
from datetime import datetime
from urllib.parse import urlparse

import numpy as np
import pandas as pd
import earthaccess
import asf_search as asf
from shapely.geometry import Point, Polygon, box
from shapely.geometry.base import BaseGeometry
from util.nisar_coherence_closure import load_station_acquisitions

log = logging.getLogger(__name__)

# --- inlined from nisar_pytools (io/search.py + utils/search_validation.py) --
# the only function from that ~5,000-line vendored package this project ever
# used was find_nisar (plus the two validators it calls) -- kept as a small
# self-contained block here instead of vendoring the whole package.

NISAR_LAUNCH = pd.Timestamp("2024-02-03")

PRODUCT_TYPES = {
    "GSLC": asf.PRODUCT_TYPE.GSLC,
    "GUNW": asf.PRODUCT_TYPE.GUNW,
    "RSLC": asf.PRODUCT_TYPE.RSLC,
    "GCOV": asf.PRODUCT_TYPE.GCOV,
    "RIFG": asf.PRODUCT_TYPE.RIFG,
    "RUNW": asf.PRODUCT_TYPE.RUNW,
    "ROFF": asf.PRODUCT_TYPE.ROFF,
    "GOFF": asf.PRODUCT_TYPE.GOFF,
    "SME2": asf.PRODUCT_TYPE.SME2,
}


def _parse_date(date, param_name):
    """Parse a date input to pd.Timestamp (naive UTC)."""
    try:
        if isinstance(date, pd.Timestamp):
            ts = date
        elif isinstance(date, np.datetime64):
            ts = pd.Timestamp(date)
        elif isinstance(date, datetime):
            ts = pd.Timestamp(date)
        elif isinstance(date, str):
            ts = pd.to_datetime(date)
        else:
            raise ValueError(
                f"{param_name} must be str, datetime, or pd.Timestamp, "
                f"got {type(date)}"
            )
    except (ValueError, pd.errors.ParserError) as e:
        raise ValueError(f"Could not parse {param_name}='{date}': {e}") from e

    if pd.isna(ts):
        raise ValueError(f"{param_name} is NaT")

    # Convert to UTC before stripping timezone to avoid offset errors
    if ts.tz is not None:
        ts = ts.tz_convert("UTC").tz_localize(None)

    return ts


def validate_dates(start_date, end_date):
    """Validate and normalize start/end dates for NISAR data search."""
    start = _parse_date(start_date, "start_date")
    end = _parse_date(end_date, "end_date")

    if start >= end:
        raise ValueError(f"start_date ({start}) must be before end_date ({end})")

    if start < NISAR_LAUNCH:
        raise ValueError(
            f"start_date ({start}) is before NISAR launch ({NISAR_LAUNCH.date()}). "
            f"No data available before this date."
        )

    return start, end


def _check_geographic_bounds(xmin, ymin, xmax, ymax):
    """Warn if coordinates look like they're not in geographic (lon/lat) range."""
    if xmin < -180 or xmax > 180 or ymin < -90 or ymax > 90:
        log.warning(
            "AOI bounds [%.1f, %.1f, %.1f, %.1f] are outside geographic "
            "coordinate range (lon: -180..180, lat: -90..90). "
            "ASF search expects geographic coordinates.",
            xmin, ymin, xmax, ymax,
        )


def validate_aoi(aoi):
    """Validate and normalize an area of interest (geometry, [xmin,ymin,xmax,ymax]
    list/tuple, (lon,lat) point, or a dict with west/south/east/north-style keys)."""
    geom = None

    if isinstance(aoi, BaseGeometry):
        geom = aoi

    elif isinstance(aoi, (list, tuple, np.ndarray)):
        if len(aoi) == 4:
            xmin, ymin, xmax, ymax = map(float, aoi)
            if xmin > xmax or ymin > ymax:
                log.warning(
                    "AOI bounds appear swapped -- auto-correcting. "
                    "Expected [xmin, ymin, xmax, ymax]."
                )
            xmin, xmax = sorted((xmin, xmax))
            ymin, ymax = sorted((ymin, ymax))
            _check_geographic_bounds(xmin, ymin, xmax, ymax)
            geom = box(xmin, ymin, xmax, ymax)
        elif len(aoi) == 2:
            geom = Point(float(aoi[0]), float(aoi[1]))
        else:
            raise ValueError(f"AOI list/tuple must have 2 or 4 elements, got {len(aoi)}")

    elif isinstance(aoi, dict):
        key_sets = [
            ("xmin", "ymin", "xmax", "ymax"),
            ("west", "south", "east", "north"),
            ("minx", "miny", "maxx", "maxy"),
        ]
        for keys in key_sets:
            if all(k in aoi for k in keys):
                xmin, ymin, xmax, ymax = (float(aoi[k]) for k in keys)
                xmin, xmax = sorted((xmin, xmax))
                ymin, ymax = sorted((ymin, ymax))
                _check_geographic_bounds(xmin, ymin, xmax, ymax)
                geom = box(xmin, ymin, xmax, ymax)
                break
        if geom is None:
            raise ValueError(f"AOI dict keys not recognized: {list(aoi.keys())}")

    else:
        raise ValueError(f"AOI must be geometry, list/tuple, or dict; got {type(aoi)}")

    if geom.is_empty:
        raise ValueError("AOI geometry is empty")
    if isinstance(geom, Polygon) and geom.area == 0:
        raise ValueError("AOI polygon has zero area")

    return geom


def _find_nisar_url_filename(url):
    """Extract the filename from a URL, stripping query strings."""
    return urlparse(url).path.rsplit("/", 1)[-1]


def find_nisar(
    aoi,
    start_date,
    end_date,
    product_type="GSLC",
    path_number=None,
    frame=None,
    direction=None,
    max_results=None,
    include_qa=False,
):
    """Search ASF for NISAR product download URLs. See nisar_pytools's
    original io/search.py docstring (git history) for full parameter docs."""
    aoi_geom = validate_aoi(aoi)
    start, end = validate_dates(start_date, end_date)

    pt_upper = product_type.upper()
    if pt_upper not in PRODUCT_TYPES:
        raise ValueError(
            f"Unknown product_type '{product_type}'. "
            f"Supported: {sorted(PRODUCT_TYPES.keys())}"
        )

    if direction is not None:
        direction_upper = direction.upper()
        if direction_upper not in ("ASCENDING", "DESCENDING"):
            raise ValueError(
                f"direction must be 'ASCENDING' or 'DESCENDING', got '{direction}'"
            )
    else:
        direction_upper = None

    search_kwargs = dict(
        platform=asf.PLATFORM.NISAR,
        intersectsWith=aoi_geom.wkt,
        start=start,
        end=end,
        processingLevel=PRODUCT_TYPES[pt_upper],
    )
    if path_number is not None:
        search_kwargs["relativeOrbit"] = path_number
    if frame is not None:
        search_kwargs["frame"] = frame
    if direction_upper is not None:
        search_kwargs["flightDirection"] = direction_upper
    if max_results is not None:
        search_kwargs["maxResults"] = max_results

    results = asf.search(**search_kwargs)
    urls = results.find_urls()

    urls = [u for u in urls if _find_nisar_url_filename(u).endswith(".h5")]
    if not include_qa:
        urls = [u for u in urls if "_QA_" not in _find_nisar_url_filename(u)]

    return urls


## 1. Station list and per-station NISAR acquisition dates

Reuses `load_station_acquisitions` (same loader `S1_burst_stations.ipynb`
and `Earsel26_NISAR-main`'s own notebooks 04/05 use) -- a station's dates are
the union across *all* of its kept NISAR tracks, matching how every other
per-station search in this project scopes its temporal window.

In [2]:
require_globals("EARSEL_NOTEBOOKS", "GSLC_WINDOWS_DIR")

stations_path = EARSEL_NOTEBOOKS / "ismn_data" / "alaska" / "stations.geojson"
with open(stations_path) as f:
    stations_geojson = json.load(f)

stations = {
    feat["properties"]["station_val"]: (
        feat["geometry"]["coordinates"][1],  # lat
        feat["geometry"]["coordinates"][0],  # lon
    )
    for feat in stations_geojson["features"]
}

nisar_dates_by_station = {}
for name in stations:
    groups = load_station_acquisitions(GSLC_WINDOWS_DIR, name)
    dates = sorted({acq["datetime"] for track_acqs in groups.values() for acq in track_acqs})
    nisar_dates_by_station[name] = dates

print(f"{len(stations)} stations")
for name, dates in nisar_dates_by_station.items():
    print(f"  {name}: {len(dates)} NISAR acquisition date(s)")


20 stations
  MooseInc: 19 NISAR acquisition date(s)
  Aniak: 0 NISAR acquisition date(s)
  AtigunPass: 18 NISAR acquisition date(s)
  Coldfoot: 19 NISAR acquisition date(s)
  EagleSummit: 28 NISAR acquisition date(s)
  GobblersKnob: 19 NISAR acquisition date(s)
  GraniteCrk: 27 NISAR acquisition date(s)
  ImnaviatCreek: 17 NISAR acquisition date(s)
  LittleChenaRidge: 29 NISAR acquisition date(s)
  McGrath: 15 NISAR acquisition date(s)
  Nenana: 0 NISAR acquisition date(s)
  NukaGlacier: 28 NISAR acquisition date(s)
  PilgrimHotSprings: 31 NISAR acquisition date(s)
  PrudhoeBay: 0 NISAR acquisition date(s)
  RockyPoint: 20 NISAR acquisition date(s)
  SpringCreek: 21 NISAR acquisition date(s)
  SusitnaValleyHigh: 25 NISAR acquisition date(s)
  TurnagainPass: 22 NISAR acquisition date(s)
  UpperNomeCreek: 29 NISAR acquisition date(s)
  Kenai-29-ENE: 18 NISAR acquisition date(s)


## 2. Search ASF for SME2 granules covering each station

Per-station 80x80m bounding box (matching the NISAR GSLC / HLS window
footprint used elsewhere in this project) and per-station temporal window
(that station's own NISAR date range, padded), searched independently and
without a track/frame/direction filter -- any relative orbit covering the
station is kept, since Alaska stations aren't restricted to a single track.
`find_nisar` only queries CMR metadata (no auth needed for search itself);
URLs are deduplicated globally before downloading, since a granule footprint
can cover more than one nearby station.

In [3]:
require_globals("stations", "nisar_dates_by_station")

WINDOW_FOOTPRINT_M = 80.0  # matches the NISAR GSLC / HLS window
_HALF_M = WINDOW_FOOTPRINT_M / 2
PAD_DAYS = pd.Timedelta(days=15)  # past the NISAR date range, same padding S1_burst_stations.ipynb uses

METERS_PER_DEG_LAT = 111_320.0


def _station_box_deg(lat, lon, half_m=_HALF_M):
    """[xmin, ymin, xmax, ymax] of a half_m-radius box around (lat, lon), in
    degrees -- longitude spacing shrinks with cos(latitude), which matters at
    Alaska's high latitudes (60-70N)."""
    dlat = half_m / METERS_PER_DEG_LAT
    dlon = half_m / (METERS_PER_DEG_LAT * np.cos(np.radians(lat)))
    return [lon - dlon, lat - dlat, lon + dlon, lat + dlat]


sme2_urls_by_station = {}
for station, (lat, lon) in stations.items():
    dates = nisar_dates_by_station.get(station, [])
    if not dates:
        print(f"{station}: no NISAR dates, skipping search")
        sme2_urls_by_station[station] = []
        continue

    aoi = _station_box_deg(lat, lon)
    search_start = min(dates) - PAD_DAYS
    search_end = max(dates) + PAD_DAYS

    urls = find_nisar(
        aoi=aoi,
        start_date=search_start,
        end_date=search_end,
        product_type="SME2",
    )
    sme2_urls_by_station[station] = urls
    print(f"{station}: {len(urls)} SME2 URL(s) found "
          f"({search_start.date()} to {search_end.date()})")

unique_urls = sorted({u for urls in sme2_urls_by_station.values() for u in urls})
print(f"\n{len(unique_urls)} unique SME2 URL(s) across all stations")


MooseInc: 22 SME2 URL(s) found (2026-06-04 to 2026-09-07)
Aniak: no NISAR dates, skipping search
AtigunPass: 24 SME2 URL(s) found (2026-06-03 to 2026-09-08)
Coldfoot: 22 SME2 URL(s) found (2026-06-03 to 2026-09-08)
EagleSummit: 41 SME2 URL(s) found (2026-06-03 to 2026-09-08)
GobblersKnob: 26 SME2 URL(s) found (2026-06-03 to 2026-09-08)
GraniteCrk: 42 SME2 URL(s) found (2026-06-03 to 2026-09-08)
ImnaviatCreek: 24 SME2 URL(s) found (2026-06-03 to 2026-09-04)
LittleChenaRidge: 43 SME2 URL(s) found (2026-06-02 to 2026-09-12)
McGrath: 21 SME2 URL(s) found (2026-06-02 to 2026-09-12)
Nenana: no NISAR dates, skipping search
NukaGlacier: 31 SME2 URL(s) found (2026-06-04 to 2026-09-12)
PilgrimHotSprings: 38 SME2 URL(s) found (2026-06-03 to 2026-09-13)
PrudhoeBay: no NISAR dates, skipping search
RockyPoint: 25 SME2 URL(s) found (2026-06-03 to 2026-09-13)
SpringCreek: 24 SME2 URL(s) found (2026-06-05 to 2026-09-12)
SusitnaValleyHigh: 27 SME2 URL(s) found (2026-06-05 to 2026-09-12)
TurnagainPass: 2

## 3. Login and download

`earthaccess.login()` authenticates against NASA Earthdata Login -- if no
cached credentials/`.netrc` are found, it will prompt for a username and
password right here in the notebook the first time (once per kernel
session). `earthaccess.download()` then fetches each URL with that
authenticated session, skipping files that already exist unless
`force=True`.

In [ ]:
require_globals("unique_urls", "SME2_OUT_DIR")

auth = earthaccess.login()
print(f"Earthdata authenticated: {auth.authenticated}")

SME2_OUT_DIR.mkdir(parents=True, exist_ok=True)

try:
    downloaded_fps = earthaccess.download(unique_urls, str(SME2_OUT_DIR), provider="ASF")
except Exception as e:
    print(f"\nDownload raised partway through: {e}")
    print("Re-run this cell to retry -- earthaccess.download() skips files already on disk.")
    downloaded_fps = [SME2_OUT_DIR / Path(u).name for u in unique_urls if (SME2_OUT_DIR / Path(u).name).exists()]

print(f"\n{len(downloaded_fps)}/{len(unique_urls)} file(s) present in {SME2_OUT_DIR}")


## 4. Manifest

`station -> [filenames]`, mirroring `S1_burst_data_alaska/station_manifest.json`'s
own convention -- so a later comparison notebook (an Alaska
`SME2_footprint_check.ipynb`/`NISAR_SM_alaska.ipynb`-style SME2 section) knows
which downloaded files are relevant to which station without re-deriving it
from filenames or re-querying ASF.

In [ ]:
require_globals("sme2_urls_by_station", "SME2_OUT_DIR")

from urllib.parse import urlparse


def _url_filename(url):
    return urlparse(url).path.rsplit("/", 1)[-1]


manifest = {
    station: sorted({_url_filename(u) for u in urls})
    for station, urls in sme2_urls_by_station.items()
}

manifest_path = SME2_OUT_DIR / "station_manifest.json"
with open(manifest_path, "w") as f:
    json.dump(manifest, f, indent=2)

n_with_data = sum(1 for files in manifest.values() if files)
print(f"{n_with_data}/{len(manifest)} station(s) have at least one SME2 file")
print(f"Wrote manifest to {manifest_path}")


<hr>

## Summary

`SME2_data_alaska/` now holds every `NISAR_L3_PR_SME2_*.h5` granule that
intersects an Alaska station's 80x80m footprint during that station's own
NISAR acquisition window, plus `station_manifest.json` mapping each station
to its relevant filenames. Nothing here samples or compares the SME2 grids
against in-situ/SAR-estimated SM yet -- that's the natural next step (an
Alaska counterpart to `SME2_footprint_check.ipynb`'s nearest-pixel lookup and
`NISAR_SM.ipynb`'s `sample_sme2()` comparison), deliberately out of scope for
this notebook, which only downloads.

Safe to re-run: Section 2's search and Section 3's `earthaccess.download()`
are both idempotent (existing files are skipped, not re-fetched), so
re-running this notebook later picks up newly available SME2 granules
without re-downloading anything already on disk.

<hr>